### Aula 07 — Joins, qualidade das chaves e planos físicos

O objetivo é integrar **clientes, pedidos e pagamentos**, acompanhando quais linhas cada join preserva e como duplicidades ou referências inexistentes afetam o resultado. A segunda frente de estudo compara `BroadcastHashJoin` e `SortMergeJoin` nos planos já registrados.

As primeiras células importam `SparkSession`, disponibilizam funções SQL como `F` e obtêm uma sessão para a aplicação. Os CSVs pequenos e os arquivos sob `large_csvs/` precisam existir nos caminhos usados pelo código; os arquivos grandes do curso podem precisar ser gerados no ambiente local.

**Como interpretar as evidências:** tabelas, contagens e planos abaixo são outputs históricos preservados; não houve nova execução. Joins, projeções e filtros constroem planos de forma lazy; `show()` e `count()` solicitam resultados. `explain(True)` expõe os planos lógicos e físico, sem pedir a saída completa. Nenhuma comparação de tempo de execução foi registrada nesta aula.

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("aula_07_joins")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/27 20:59:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/27 20:59:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


#### 1. Fontes e chaves da integração

As leituras criam `df_clientes` e `df_pedidos` com cabeçalho e inferência de tipos, que pode demandar leitura. `cliente_id` relaciona o cadastro de clientes aos pedidos; `pedido_id` identificará o vínculo com pagamentos lidos adiante.

Uma igualdade de chave informa ao Spark quais linhas podem combinar; ela **não impõe unicidade nem integridade referencial** aos CSVs. Antes de consumir métricas, é necessário verificar duplicidades e referências sem cadastro. Um cliente pode ter vários pedidos, e um pedido pode ter zero, um ou vários registros de pagamento: essas relações precisam ser observadas, não pressupostas.

In [3]:
df_clientes = spark.read.csv("/workspace/datasets/07_joins/clientes.csv", header= True, inferSchema= True)
df_pedidos = spark.read.csv("/workspace/datasets/07_joins/pedidos.csv", header= True, inferSchema= True)

#### 2. `inner`: combinar apenas correspondências

`df_clientes.join(df_pedidos, on="cliente_id", how="inner")` retorna combinações em que a chave coincide. Clientes sem pedidos e pedidos sem cliente correspondente ficam fora. Se um cliente tiver vários pedidos, ele aparece uma vez para cada correspondência.

A forma `on="cliente_id"` usa o nome existente nos dois lados e apresenta uma só coluna de chave no resultado. A projeção seguinte mantém `cliente_id`, `pedido_id` e `nome`; `show()` exibe os primeiros registros necessários à amostra. **Output salvo:** há linhas de pedidos associados aos nomes dos clientes, limitadas a 20; isso não é a contagem total da integração nem uma ordenação garantida por pedido.

In [4]:
df_inner = df_clientes.join(
    df_pedidos,
    on= "cliente_id",
    how= "inner"
)

In [5]:
df_join_inner = (
    df_inner.select("cliente_id", "pedido_id", "nome")
)

In [6]:
df_join_inner.show()

+----------+---------+----------------+
|cliente_id|pedido_id|            nome|
+----------+---------+----------------+
|       452|    10001| Mariana Barbosa|
|        71|    10002|  Olivia Almeida|
|       411|    10003|   Eduarda Costa|
|       184|    10004|   Bruno Almeida|
|       290|    10005|   Tatiana Costa|
|       164|    10006|    Carla Mendes|
|       242|    10007| Olivia Carvalho|
|        52|    10008|     Bruno Silva|
|       406|    10009|   Diego Barbosa|
|       409|    10010|    Olivia Costa|
|       304|    10011|   Nicolas Rocha|
|        15|    10012|Henrique Martins|
|        97|    10013|   Diego Martins|
|        47|    10014|  Henrique Costa|
|       175|    10016|     Diego Gomes|
|        76|    10017|   Nicolas Souza|
|       434|    10018|  Bruno Carvalho|
|       340|    10019|    Lucas Santos|
|        70|    10020|  Samuel Ribeiro|
|       250|    10021| Tatiana Martins|
+----------+---------+----------------+
only showing top 20 rows



#### 3. `left`: preservar clientes sem pedidos

O `left` mantém todas as linhas de `df_clientes`, combinando pedidos quando há correspondência. Para os clientes sem correspondência, as colunas vindas de pedidos ficam `NULL`. Havendo vários pedidos, uma linha de cliente ainda pode originar várias linhas: preservar a esquerda não significa preservar sua quantidade de linhas.

O filtro `pedido_id.isNull()` isola as linhas cujo identificador de pedido ficou ausente. **Amostra salva:** aparecem clientes como `471` e `472` com `pedido_id = NULL`.

Usar um campo da direita como indicador de ausência exige que ele não seja nulo em registros realmente correspondentes; caso contrário, pode misturar ausência de match e dado faltante. Para testar apenas a existência da chave, o próximo `left_anti` expressa essa intenção diretamente.

In [7]:
df_left = df_clientes.join(
    df_pedidos,
    on= "cliente_id",
    how= "left"
)

In [8]:
df_left_join = (
    df_left.select("cliente_id", "pedido_id", "nome")
    .filter(F.col("pedido_id").isNull())
)

In [9]:
df_left_join.show()

+----------+---------+----------------+
|cliente_id|pedido_id|            nome|
+----------+---------+----------------+
|       471|     NULL|   Karen Barbosa|
|       472|     NULL|   Paulo Almeida|
|       473|     NULL|  Lucas Oliveira|
|       474|     NULL|Henrique Martins|
|       475|     NULL|   Bruno Almeida|
|       476|     NULL|  Henrique Costa|
|       477|     NULL|     Diego Rocha|
|       478|     NULL|  Mariana Santos|
|       479|     NULL|     Carla Rocha|
|       480|     NULL|  Vinicius Souza|
|       481|     NULL|   Eduarda Silva|
|       482|     NULL|     Karen Silva|
|       483|     NULL|     Diego Souza|
|       484|     NULL|  Isabela Santos|
|       485|     NULL|   Lucas Almeida|
|       486|     NULL|  Mariana Santos|
|       487|     NULL|       João Lima|
|       488|     NULL|   Karen Almeida|
|       489|     NULL|  Eduarda Santos|
|       490|     NULL|Gabriela Almeida|
+----------+---------+----------------+
only showing top 20 rows



#### 4. `left_anti`: clientes que não possuem pedido correspondente

`left_anti` retorna somente linhas e colunas da esquerda para as quais não existe correspondência à direita. Aqui a esquerda é `df_clientes`, portanto a pergunta é: **quais clientes não aparecem nos pedidos?** Não são retornadas colunas de pedidos preenchidas com nulos.

A amostra salva começa pelos mesmos clientes sem pedidos vistos no filtro do left join, agora com as colunas do cadastro. Colocar outra tabela à esquerda muda a pergunta: pedidos à esquerda permitirão procurar pedidos sem cliente. Com a igualdade comum de chaves, valores nulos não se correspondem, o que também deve ser considerado na avaliação de qualidade.

In [10]:
df_left_anti = df_clientes.join(
    df_pedidos,
    on= "cliente_id",
    how= "left_anti"
)

In [11]:
df_left_anti.show()

+----------+----------------+--------------------+------+----------+-------------+-----+
|cliente_id|            nome|              cidade|estado|  segmento|data_cadastro|ativo|
+----------+----------------+--------------------+------+----------+-------------+-----+
|       471|   Karen Barbosa|            Salvador|    BA|    Varejo|   2024-12-08|false|
|       472|   Paulo Almeida|            Curitiba|    PR|    Varejo|   2022-04-24| true|
|       473|  Lucas Oliveira|              Olinda|    PE|    Varejo|   2022-07-30| true|
|       474|Henrique Martins|            Curitiba|    PR|    Varejo|   2022-04-01| true|
|       475|   Bruno Almeida|    Feira de Santana|    BA|    Varejo|   2022-11-04| true|
|       476|  Henrique Costa|            Camaçari|    BA|    Varejo|   2023-10-18| true|
|       477|     Diego Rocha|Vitória da Conquista|    BA|       PME|   2022-04-18| true|
|       478|  Mariana Santos|              Recife|    PE|       PME|   2023-08-02| true|
|       479|     Carl

#### 5. `left_semi`: clientes que possuem algum pedido

`left_semi` mantém apenas as linhas e colunas de clientes que encontram pelo menos um pedido. Diferentemente do `inner`, não carrega atributos de pedidos e **não repete uma linha da esquerda por cada pedido encontrado**; testa existência.

A saída salva apresenta clientes com pedidos, como os IDs `1` a `20` na amostra. O semi join não deduplica a própria esquerda: se o cadastro já contivesse linhas repetidas, essas linhas ainda poderiam ser retornadas. Essa operação é útil para selecionar uma população válida antes de enriquecer seus dados em outro passo.

In [12]:
df_left_semi = df_clientes.join(
    df_pedidos,
    on= "cliente_id",
    how= "left_semi"
)

In [13]:
df_left_semi.show()

+----------+----------------+--------------------+------+----------+-------------+-----+
|cliente_id|            nome|              cidade|estado|  segmento|data_cadastro|ativo|
+----------+----------------+--------------------+------+----------+-------------+-----+
|         1|     Ana Martins|            Curitiba|    PR|    Varejo|   2023-04-03| true|
|         2| Tatiana Pereira|            Salvador|    BA|    Varejo|   2022-07-11| true|
|         3|    Samuel Souza|              Recife|    PE|       PME|   2025-01-21| true|
|         4|     Ana Barbosa|            Contagem|    MG|       PME|   2024-05-15| true|
|         5|    Karen Santos|            Campinas|    SP|    Varejo|   2022-07-18| true|
|         6|   Bruno Martins|          Petrópolis|    RJ|    Varejo|   2022-09-13| true|
|         7|     Samuel Lima|      Belo Horizonte|    MG|       PME|   2025-06-20| true|
|         8|     Carla Silva|             Niterói|    RJ|       PME|   2023-08-16| true|
|         9|    Maria

#### 6. Validar cardinalidade antes de calcular métricas

O primeiro diagnóstico agrupa clientes por `cliente_id`, conta ocorrências e mantém contagens maiores que um. **Resultado salvo:** a tabela de duplicidades está vazia para os IDs contados.

A expressão usada é `count("cliente_id")`, que ignora valores nulos. Portanto, esse diagnóstico trata repetições de IDs não nulos, mas não comprova ausência de clientes com chave nula. Unicidade de uma chave e relação 1:1 entre duas tabelas são propriedades diferentes: a segunda exige avaliar correspondências nos dois lados.

Em Engenharia de Dados, essa verificação protege a integração contra duplicação involuntária de pedidos e contra métricas infladas por um cadastro com várias linhas para a mesma chave.

In [14]:
df_clientes_validacao = (
    df_clientes.groupBy("cliente_id")
    .agg(F.count("cliente_id").alias("quantidade_clientes"))
    .filter(F.col("quantidade_clientes") > 1)
    
    
)

In [15]:
df_clientes_validacao.show()

+----------+-------------------+
|cliente_id|quantidade_clientes|
+----------+-------------------+
+----------+-------------------+



##### 6.1. Vários pedidos por cliente e multiplicação de linhas

`df_pedidos_validacao` conta todas as linhas de pedidos por `cliente_id`, usando `count("*")`. A amostra registra, por exemplo, `19` pedidos para o cliente `243` e `13` para `148`: a relação observada entre um cadastro único e seus pedidos é 1:N.

Em um join por igualdade, se uma chave tiver **m linhas na esquerda e n na direita**, suas correspondências geram **m × n linhas**. Com uma ocorrência em cada lado, temos 1:1; com repetição dos dois lados, N:N. Somar `valor_total` após outro join pode repetir o valor de um mesmo pedido por cada correspondência adicional.

O diagnóstico inclui também o ID `516`, que será visto entre os pedidos órfãos. Contar pedidos por cliente não prova que esses clientes existam no cadastro; cardinalidade e integridade são verificações complementares. A distribuição desigual de chaves pode concentrar trabalho em joins por shuffle, mas esta amostra não comprova um problema de skew nem seu impacto.

In [16]:
df_pedidos_validacao = (
    df_pedidos.groupBy("cliente_id")
    .agg(F.count("*").alias("quantidade_pedidos"))
)

In [17]:
df_pedidos_validacao.show()

+----------+------------------+
|cliente_id|quantidade_pedidos|
+----------+------------------+
|       463|                 7|
|       148|                13|
|       243|                19|
|       392|                 8|
|       516|                12|
|        31|                10|
|        85|                11|
|       137|                 9|
|       251|                 8|
|       451|                 7|
|       458|                11|
|        65|                 6|
|        53|                14|
|       255|                10|
|       133|                13|
|       296|                10|
|        78|                16|
|       513|                 6|
|       322|                10|
|       321|                10|
+----------+------------------+
only showing top 20 rows



#### 7. Pedidos órfãos: inverter a tabela investigada

Agora `df_pedidos` fica à esquerda de um `left_anti` contra clientes. O resultado contém pedidos cuja chave não encontra cadastro correspondente, uma verificação de integridade referencial.

**Resultados salvos:** o `show()` apresenta exemplos, incluindo cliente `516` no pedido `10015`; o `count()` retorna **166 registros de pedidos órfãos**. O total é de linhas de pedidos, não de clientes distintos. `show()` e `count()` são actions separadas e podem disparar trabalho separado, pois não há cache explícito neste trecho.

Esses registros não participam do `inner` entre clientes e pedidos. O código apenas os identifica; não corrige nem remove dados da fonte.

In [18]:
df_pedidos_orfaos = df_pedidos.join(
    df_clientes,
    on= "cliente_id",
    how= "left_anti"
)

In [19]:
df_pedidos_orfaos.show()
df_pedidos_orfaos.count()

+----------+---------+-----------+---------+-----------+-----------+
|cliente_id|pedido_id|data_pedido|   status|      canal|valor_total|
+----------+---------+-----------+---------+-----------+-----------+
|       516|    10015| 2025-01-01|concluido|        app|     219.12|
|       506|    10044| 2025-11-23|concluido|       site|     201.01|
|       518|    10109| 2025-04-13|cancelado|       site|     514.55|
|       508|    10117| 2025-03-09|concluido|       site|     167.81|
|       503|    10129| 2025-02-26|  enviado|       loja|     479.25|
|       506|    10147| 2025-09-16|  enviado|       loja|     597.05|
|       512|    10158| 2025-11-21|  enviado|marketplace|       20.0|
|       515|    10188| 2025-12-30|  enviado|       site|     422.19|
|       508|    10226| 2025-04-24|cancelado|       loja|     427.25|
|       502|    10251| 2025-03-07|concluido|        app|      669.3|
|       501|    10269| 2025-10-23|concluido|       site|    1087.79|
|       517|    10345| 2025-03-11|

166

#### 8. O `inner` pequeno no plano físico: broadcast de clientes

`df_inner.explain(True)` detalha a consulta antes da projeção de três colunas criada em `df_join_inner`. Por isso, o plano mantém mais atributos do que aquela amostra mostrada no início.

O plano salvo contém **`BroadcastHashJoin ..., Inner, BuildLeft`**. Neste join, clientes são a entrada esquerda: o `BroadcastExchange` prepara sua relação hash e a distribui para os executors que processam o outro lado. As tasks que leem pedidos procuram as chaves nessa estrutura, evitando redistribuir os dois lados por hash e ordená-los para o join.

`BuildLeft` identifica o lado usado para construir a estrutura hash; não significa que a semântica seja um left join. Aqui a semântica continua `Inner`. Os `Filter isnotnull(cliente_id)` são compatíveis com igualdade comum: chaves nulas não geram correspondências no inner.

`AdaptiveSparkPlan isFinalPlan=false` mostra uma estratégia planejada ainda não finalizada pelo AQE. Broadcast depende de elegibilidade, tamanho em bytes estimado, estatísticas, configurações e memória disponível; a quantidade de linhas sozinha não explica a escolha. Este plano não contém uma medição de quanto tempo ou rede foi economizado.

In [20]:
df_inner.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [cliente_id])
:- Relation [cliente_id#17,nome#18,cidade#19,estado#20,segmento#21,data_cadastro#22,ativo#23] csv
+- Relation [pedido_id#48,cliente_id#49,data_pedido#50,status#51,canal#52,valor_total#53] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, cidade: string, estado: string, segmento: string, data_cadastro: date, ativo: boolean, pedido_id: int, data_pedido: date, status: string, canal: string, valor_total: double
Project [cliente_id#17, nome#18, cidade#19, estado#20, segmento#21, data_cadastro#22, ativo#23, pedido_id#48, data_pedido#50, status#51, canal#52, valor_total#53]
+- Join Inner, (cliente_id#17 = cliente_id#49)
   :- Relation [cliente_id#17,nome#18,cidade#19,estado#20,segmento#21,data_cadastro#22,ativo#23] csv
   +- Relation [pedido_id#48,cliente_id#49,data_pedido#50,status#51,canal#52,valor_total#53] csv

== Optimized Logical Plan ==
Project [cliente_id#17, nome#18, cidade#19, estado#20, segmento#21, data_c

#### 9. Duas fontes maiores: preparar o contraste

As próximas células leem `clientes_large.csv` e `pedidos_large.csv`, constroem outro `inner` por `cliente_id` e inspecionam seu plano. A semântica das correspondências é a mesma; o objetivo é observar outra estratégia física com fontes diferentes.

O sufixo `large` não define uma regra da API. A escolha do operador depende das propriedades e estatísticas dos dados, do tipo de join, de hints e das configurações. Os outputs existentes demonstram a estratégia selecionada naquele ambiente, sem informar aqui as contagens totais ou o tamanho em bytes dessas fontes.

In [21]:
df_clientes_large = spark.read.csv("/workspace/datasets/07_joins/large_csvs/clientes_large.csv", header= True, inferSchema= True)
df_pedidos_large = spark.read.csv("/workspace/datasets/07_joins/large_csvs/pedidos_large.csv", header= True, inferSchema= True)

In [22]:
df_inner_large = df_clientes_large.join(
    df_pedidos_large,
    on= "cliente_id",
    how= "inner"
)

##### 9.1. `SortMergeJoin`: dois Exchanges e duas ordenações

No plano salvo de `df_inner_large`, o Spark escolheu **`SortMergeJoin ..., Inner`**. Leia cada ramo de baixo para cima:

1. `FileScan csv` fornece as linhas, e `Filter isnotnull(cliente_id)` descarta chaves que não correspondem no inner.
2. Cada **`Exchange hashpartitioning(cliente_id, 200)`** redistribui uma entrada pela chave. Há dois Exchanges porque as duas entradas precisam de distribuições compatíveis para encontrar as correspondências.
3. Cada **`Sort [cliente_id ASC NULLS FIRST], false, 0`** ordena dentro de suas partições; `false` não representa uma ordenação global do resultado.
4. **`SortMergeJoin`** percorre as entradas ordenadas, combina chaves iguais e produz seus pares correspondentes.

O shuffle movimenta dados; o sort ordena; o join combina. O `200` é o particionamento de shuffle planejado, sem garantir 200 tasks simultâneas. O AQE aparece com `isFinalPlan=false`, portanto pode adaptar o plano durante a execução. Distribuições muito desiguais por chave podem fazer algumas tasks trabalharem mais, mas não há métricas de tasks nesta saída para diagnosticar isso.

In [23]:
df_inner_large.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [cliente_id])
:- Relation [cliente_id#329,cidade#330,estado#331,segmento#332,data_cadastro#333,ativo#334] csv
+- Relation [pedido_id#358,cliente_id#359,data_pedido#360,status#361,canal#362,valor_total#363] csv

== Analyzed Logical Plan ==
cliente_id: int, cidade: string, estado: string, segmento: string, data_cadastro: date, ativo: boolean, pedido_id: int, data_pedido: date, status: string, canal: string, valor_total: double
Project [cliente_id#329, cidade#330, estado#331, segmento#332, data_cadastro#333, ativo#334, pedido_id#358, data_pedido#360, status#361, canal#362, valor_total#363]
+- Join Inner, (cliente_id#329 = cliente_id#359)
   :- Relation [cliente_id#329,cidade#330,estado#331,segmento#332,data_cadastro#333,ativo#334] csv
   +- Relation [pedido_id#358,cliente_id#359,data_pedido#360,status#361,canal#362,valor_total#363] csv

== Optimized Logical Plan ==
Project [cliente_id#329, cidade#330, estado#331, segmento#332, data_cadastro

#### 10. Pedidos grandes com clientes pequenos e o hint isolado

`df_broadcast` combina `df_pedidos_large` à esquerda com `df_clientes` à direita. O código constrói o join **antes** da célula que chama `F.broadcast(df_clientes)` separadamente.

`F.broadcast(...)` devolve um DataFrame com um hint de broadcast; a chamada isolada não altera retroativamente o plano de `df_broadcast`, nem substitui a variável `df_clientes`. Seu output salvo é apenas a representação de um DataFrame, não os dados nem uma medição de transmissão. Para um hint influenciar uma consulta, o DataFrame retornado precisa participar da construção dessa consulta.

Assim, o próximo plano permite observar uma escolha de broadcast feita para `df_broadcast`, mas não demonstra que a chamada isolada a tenha causado. Este detalhe deve ser considerado ao comparar intenção do experimento e evidência do arquivo.

In [24]:
df_broadcast = df_pedidos_large.join(
    df_clientes,
    on= "cliente_id",
    how= "inner"
)

In [25]:
F.broadcast(df_clientes)

DataFrame[cliente_id: int, nome: string, cidade: string, estado: string, segmento: string, data_cadastro: date, ativo: boolean]

##### 10.1. `BuildRight` no plano observado

O output de `df_broadcast.explain(True)` contém **`BroadcastHashJoin ..., Inner, BuildRight`** e um `BroadcastExchange` no ramo de clientes. Aqui clientes são a entrada **direita**, ao contrário do primeiro inner pequeno, em que eram a esquerda.

A estrutura hash continua sendo construída com os clientes pequenos; quem mudou foi sua posição neste join. As partições de pedidos grandes percorrem a relação e consultam essa estrutura localmente. O plano não mostra o shuffle bilateral e os sorts do exemplo anterior, mas ainda existe transferência do lado transmitido por broadcast e consumo de memória para mantê-lo.

A vantagem potencial depende do tamanho real desse lado e dos recursos; não há tempos registrados para afirmar ganho quantitativo.

In [26]:
df_broadcast.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [cliente_id])
:- Relation [pedido_id#358,cliente_id#359,data_pedido#360,status#361,canal#362,valor_total#363] csv
+- Relation [cliente_id#17,nome#18,cidade#19,estado#20,segmento#21,data_cadastro#22,ativo#23] csv

== Analyzed Logical Plan ==
cliente_id: int, pedido_id: int, data_pedido: date, status: string, canal: string, valor_total: double, nome: string, cidade: string, estado: string, segmento: string, data_cadastro: date, ativo: boolean
Project [cliente_id#359, pedido_id#358, data_pedido#360, status#361, canal#362, valor_total#363, nome#18, cidade#19, estado#20, segmento#21, data_cadastro#22, ativo#23]
+- Join Inner, (cliente_id#359 = cliente_id#17)
   :- Relation [pedido_id#358,cliente_id#359,data_pedido#360,status#361,canal#362,valor_total#363] csv
   +- Relation [cliente_id#17,nome#18,cidade#19,estado#20,segmento#21,data_cadastro#22,ativo#23] csv

== Optimized Logical Plan ==
Project [cliente_id#359, pedido_id#358, data_pedido#360

#### 11. Aliases e seleção de colunas com nomes repetidos

`alias("cli")` e `alias("ped")` qualificam as entradas para que `F.col("cli.cliente_id")`, `F.col("cli.estado")` e `F.col("ped.pedido_id")` indiquem a origem desejada. Aliases não criam cópias materiais das tabelas nem executam o join.

Este bloco usa uma condição explícita `clientes.cliente_id == pedidos.cliente_id`. Diferentemente de `on="cliente_id"`, essa forma pode manter ambas as colunas de chave no resultado; qualificar as referências evita ambiguidades. `df_clientes_pedidos_final` prepara uma projeção de cliente/estado e pedido/data/valor, sem action nem output de dados neste bloco.

In [27]:
clientes = df_clientes.alias("cli")
pedidos = df_pedidos.alias("ped")

In [28]:
df_clientes_pedidos = clientes.join(
    pedidos,
    clientes.cliente_id == pedidos.cliente_id,
    "inner"
)

In [29]:
df_clientes_pedidos_final = (
    df_clientes_pedidos.select(
        F.col("cli.cliente_id"),
        F.col("cli.estado"),
        F.col("ped.pedido_id"),
        F.col("ped.data_pedido"),
        F.col("ped.valor_total")
    )
)

#### 12. Adicionar a fonte de pagamentos

A leitura de `pagamentos.csv` cria `df_pagamentos`, cuja chave de integração com pedidos é `pedido_id`. `show(1)` apresenta somente um registro: no output salvo, o pagamento `90001` está associado ao pedido `10001`.

Essa amostra mostra os campos disponíveis, incluindo `status_pagamento` e `valor_pago`; não comprova unicidade por pedido nem pagamento integral de todos os pedidos. O pipeline seguinte seleciona alguns atributos de pagamentos, sem filtrar por status ou calcular uma conciliação financeira.

In [30]:
df_pagamentos = spark.read.csv("/workspace/datasets/07_joins/pagamentos.csv", header= True, inferSchema= True)
df_pagamentos.show(1)

+------------+---------+---------------+----------------+----------+--------------+
|pagamento_id|pedido_id|forma_pagamento|status_pagamento|valor_pago|data_pagamento|
+------------+---------+---------------+----------------+----------+--------------+
|       90001|    10001|  Cartão Débito|        aprovado|    473.56|    2025-10-12|
+------------+---------+---------------+----------------+----------+--------------+
only showing top 1 row



#### 13. Integrar três tabelas com preservação do resultado intermediário

Os aliases `cli`, `ped` e `pag` identificam as três origens. A sequência preservada no código é:

`clientes INNER pedidos por cliente_id → resultado LEFT pagamentos por pedido_id → seleção de colunas`.

O primeiro join exclui pedidos sem cliente correspondente. O segundo preserva **as linhas desse resultado intermediário**, inclusive pedidos que não encontram pagamento; as colunas de pagamento ficam nulas nesses casos. Portanto, ele não recupera os pedidos órfãos descartados pelo primeiro inner.

A projeção final escolhe `cliente_id`, `estado`, `pedido_id`, `valor_total`, `pagamento_id`, `forma_pagamento` e `valor_pago`. **Exemplo salvo:** o pedido `10006` permanece com seus atributos de cliente/pedido e sem valores de pagamento. Se houvesse vários pagamentos correspondentes ao mesmo pedido, esse left join poderia multiplicar a linha do pedido.

In [31]:
clientes = df_clientes.alias("cli")
pedidos = df_pedidos.alias("ped")
pagamentos = df_pagamentos.alias("pag")

In [32]:
df_clientes_pedidos = clientes.join(
    pedidos,
    clientes.cliente_id == pedidos.cliente_id,
    "inner"
)

In [33]:
df_clientes_pedidos_pagamentos = df_clientes_pedidos.join(
    pagamentos,
    df_clientes_pedidos.pedido_id == pagamentos.pedido_id,
    "left"
)

In [34]:
df_clientes_pedidos_pagamentos_final = (
    df_clientes_pedidos_pagamentos.select(
        F.col("cli.cliente_id"),
        F.col("cli.estado"),
        F.col("ped.pedido_id"),
        F.col("ped.valor_total"),
        F.col("pag.pagamento_id"),
        F.col("pag.forma_pagamento"),
        F.col("pag.valor_pago")
    )
)

In [35]:
df_clientes_pedidos_pagamentos_final.show()

+----------+------+---------+-----------+------------+---------------+----------+
|cliente_id|estado|pedido_id|valor_total|pagamento_id|forma_pagamento|valor_pago|
+----------+------+---------+-----------+------------+---------------+----------+
|       452|    RJ|    10001|     473.56|       90001|  Cartão Débito|    473.56|
|        71|    PE|    10002|     303.13|       90002|            Pix|    303.13|
|       411|    BA|    10003|     281.64|       90003|            Pix|    281.64|
|       184|    MG|    10004|     281.65|       90004| Cartão Crédito|    281.65|
|       290|    SP|    10005|     893.22|       90005|            Pix|    893.22|
|       164|    PR|    10006|     562.28|        NULL|           NULL|      NULL|
|       242|    SP|    10007|     233.26|       90006|         Boleto|    233.26|
|        52|    MG|    10008|     487.87|       90007| Cartão Crédito|    487.87|
|       406|    PE|    10009|     399.09|       90008| Cartão Crédito|    399.09|
|       409|    

##### 13.1. Investigar pagamentos ausentes no pipeline

O filtro `pagamento_id.isNull()` seleciona linhas preservadas pelo left join cujo identificador de pagamento ficou nulo. A amostra salva inclui os pedidos `10006`, `10040` e outros, com os três campos de pagamento nulos.

Esse diagnóstico deve ser lido junto com a qualidade do próprio identificador: um `pagamento_id` nulo na fonte também poderia produzir uma linha filtrada, mesmo com correspondência por pedido. Ele é diferente de procurar **pagamentos órfãos**, pois aqui a investigação parte dos pedidos válidos do pipeline.

In [36]:
df_clientes_pedidos_pagamentos_final_nulos = (
    df_clientes_pedidos_pagamentos_final.filter(
        F.col("pagamento_id").isNull()
    )
)

In [37]:
df_clientes_pedidos_pagamentos_final_nulos.show()

+----------+------+---------+-----------+------------+---------------+----------+
|cliente_id|estado|pedido_id|valor_total|pagamento_id|forma_pagamento|valor_pago|
+----------+------+---------+-----------+------------+---------------+----------+
|       164|    PR|    10006|     562.28|        NULL|           NULL|      NULL|
|       370|    RJ|    10040|     292.38|        NULL|           NULL|      NULL|
|        60|    MG|    10045|     327.27|        NULL|           NULL|      NULL|
|       414|    SP|    10050|      431.7|        NULL|           NULL|      NULL|
|       352|    MG|    10052|     227.09|        NULL|           NULL|      NULL|
|       388|    PR|    10055|     360.44|        NULL|           NULL|      NULL|
|       117|    SP|    10068|     329.52|        NULL|           NULL|      NULL|
|       193|    RJ|    10084|     262.86|        NULL|           NULL|      NULL|
|       133|    MG|    10092|     414.46|        NULL|           NULL|      NULL|
|       100|    

#### 14. Comparar quantidades antes e depois dos pagamentos

As duas próximas actions retornam **4.834 linhas**: tanto para `df_clientes_pedidos` quanto para `df_clientes_pedidos_pagamentos`. São contagens completas registradas, ao contrário das amostras de 20 linhas.

Como o segundo join é left e não remove linhas intermediárias, a igualdade das contagens é compatível com não haver múltiplos pagamentos correspondentes a uma mesma linha desse intermediário. Não comprova que todos os pedidos foram pagos, pois uma linha sem match também é preservada uma vez. Também não comprova unicidade de `pedido_id` na tabela inteira de pagamentos; a verificação seguinte investiga essa questão.

Comparar contagens e chaves em pontos de integração ajuda a detectar multiplicações antes de somar faturamento ou quantidade de pedidos em uma tabela preparada para consumo analítico.

In [38]:
df_clientes_pedidos.count()

4834

In [39]:
df_clientes_pedidos_pagamentos.count()

4834

##### 14.1. Duplicidades que precisam encontrar correspondência

`df_pagamentos_validacao` agrupa pagamentos por `pedido_id`, conta valores não nulos dessa chave e retorna contagens maiores que um. O alias `quantidade_pedidos` é o nome usado pelo código; a métrica conta registros de pagamentos por ID de pedido.

**Resultado salvo:** os IDs `20082` e `20001` têm duas ocorrências cada. Eles também aparecem no diagnóstico de pagamentos órfãos logo adiante. Por não encontrarem pedidos, essas duplicidades não multiplicam o pipeline de pedidos válidos, o que explica a compatibilidade com as duas contagens de `4834`.

Em outro conjunto, duplicidades que encontrassem uma chave do intermediário poderiam repetir `valor_total` e inflar uma soma posterior. A ocorrência de repetição por chave e sua participação efetiva no join são evidências diferentes. Como antes, `count("pedido_id")` não diagnostica registros cuja própria chave seja nula.

In [40]:
df_pagamentos_validacao = (
    df_pagamentos.groupBy("pedido_id")
    .agg(F.count("pedido_id").alias("quantidade_pedidos"))
    .filter(F.col("quantidade_pedidos") > 1)
)

In [41]:
df_pagamentos_validacao.show()

+---------+------------------+
|pedido_id|quantidade_pedidos|
+---------+------------------+
|    20082|                 2|
|    20001|                 2|
+---------+------------------+



#### 15. Pagamentos órfãos: integridade por `pedido_id`

`pagamentos.join(pedidos, ..., "left_anti")` retorna pagamentos que não encontram pedido pela chave. A tabela desejada está à esquerda, independentemente de a igualdade no código escrever a referência de pedidos antes da referência de pagamentos.

**Resultados salvos:** `count()` retorna **25 registros de pagamentos órfãos**; o `show()` apresenta uma amostra de 20, incluindo as ocorrências repetidas de `20082` e `20001`. Não são necessariamente 25 IDs de pedido distintos.

A comparação é contra todos os pedidos lidos, não apenas os pedidos que sobreviveram ao inner com clientes. Portanto, um pagamento de pedido existente com cliente inválido não seria classificado como órfão por esta checagem, ainda que fique fora do pipeline final. Cada regra de qualidade precisa explicitar o conjunto de referência utilizado.

In [42]:
df_pagamentos_orfaos = pagamentos.join(
    pedidos,
    pedidos.pedido_id == pagamentos.pedido_id,
    "left_anti"
)

In [43]:
df_pagamentos_orfaos.count()

25

In [44]:
df_pagamentos_orfaos.show()

+------------+---------+---------------+----------------+----------+--------------+
|pagamento_id|pedido_id|forma_pagamento|status_pagamento|valor_pago|data_pagamento|
+------------+---------+---------------+----------------+----------+--------------+
|       94611|    20068|            Pix|        aprovado|    611.01|    2025-01-08|
|       94612|    20084| Cartão Crédito|        aprovado|   1325.94|    2025-05-08|
|       94613|    20039|            Pix|        aprovado|    756.44|    2025-12-25|
|       94614|    20045|  Cartão Débito|        aprovado|    640.71|    2025-05-15|
|       94615|    20012|            Pix|        aprovado|    984.86|    2025-10-24|
|       94616|    20083|            Pix|        aprovado|     97.27|    2025-02-21|
|       94617|    20037|         Boleto|        aprovado|   1303.33|    2025-10-18|
|       94618|    20082|            Pix|        aprovado|    962.57|    2025-10-23|
|       94619|    20064| Cartão Crédito|        aprovado|   1413.95|    2025

#### 16. Plano do pipeline completo: dois broadcasts e poda de colunas

O plano de `df_clientes_pedidos_pagamentos_final` mostra a mesma sequência lógica de integração, com duas estratégias físicas específicas:

- **Inner por `cliente_id`, `BuildLeft`**: transmite clientes para construir a estrutura hash e combina com pedidos. O `FileScan` de clientes lê somente `cliente_id` e `estado`; o de pedidos, `pedido_id`, `cliente_id` e `valor_total`.
- **LeftOuter por `pedido_id`, `BuildRight`**: transmite pagamentos, a entrada direita deste segundo join, para enriquecer o intermediário preservado à esquerda. Seu `FileScan` contém `pagamento_id`, `pedido_id`, `forma_pagamento` e `valor_pago`.

Cada `BroadcastExchange` pertence a um join; `BuildLeft/BuildRight` deve ser interpretado em relação às entradas desse operador específico. A transmissão do lado direito no left join é compatível com manter as linhas do intermediário quando a busca não encontra pagamento.

Os `Project` e `ReadSchema` menores mostram **Column Pruning**: atributos de fontes que não chegam à projeção nem servem de chave são dispensados do schema lido. `Filter isnotnull(pedido_id)` no ramo de pagamentos não elimina pedidos sem pagamento do resultado, pois filtra a entrada direita antes do left join. O plano salva `isFinalPlan=false`; ele descreve a estratégia exibida, sem métricas finais de execução.

In [45]:
df_clientes_pedidos_pagamentos_final.explain(True)

== Parsed Logical Plan ==
'Project ['cli.cliente_id, 'cli.estado, 'ped.pedido_id, 'ped.valor_total, 'pag.pagamento_id, 'pag.forma_pagamento, 'pag.valor_pago]
+- Join LeftOuter, (pedido_id#48 = pedido_id#444)
   :- Join Inner, (cliente_id#17 = cliente_id#49)
   :  :- SubqueryAlias cli
   :  :  +- Relation [cliente_id#17,nome#18,cidade#19,estado#20,segmento#21,data_cadastro#22,ativo#23] csv
   :  +- SubqueryAlias ped
   :     +- Relation [pedido_id#48,cliente_id#49,data_pedido#50,status#51,canal#52,valor_total#53] csv
   +- SubqueryAlias pag
      +- Relation [pagamento_id#443,pedido_id#444,forma_pagamento#445,status_pagamento#446,valor_pago#447,data_pagamento#448] csv

== Analyzed Logical Plan ==
cliente_id: int, estado: string, pedido_id: int, valor_total: double, pagamento_id: int, forma_pagamento: string, valor_pago: double
Project [cliente_id#17, estado#20, pedido_id#48, valor_total#53, pagamento_id#443, forma_pagamento#445, valor_pago#447]
+- Join LeftOuter, (pedido_id#48 = pedido_

#### 17. Encerramento

`spark.stop()` encerra os recursos da sessão/contexto. Os DataFrames dependem de uma sessão ativa para novas actions; suas tabelas e planos salvos permanecem como registro do experimento.

In [46]:
spark.stop()

#### Resumo dos aprendizados

| Pergunta | Operação usada | Evidência salva nesta aula |
|---|---|---|
| Quais clientes não têm pedidos? | Clientes `left_anti` pedidos | Amostra de clientes sem correspondência |
| Quais clientes têm algum pedido? | Clientes `left_semi` pedidos | Apenas colunas de clientes, sem multiplicação por pedido |
| Quais pedidos não encontram cliente? | Pedidos `left_anti` clientes | 166 registros |
| O left de pagamentos multiplicou o intermediário? | Contagem antes e depois | 4.834 → 4.834 linhas |
| Existem repetições por pedido em pagamentos? | Agrupamento e filtro de contagem | IDs 20082 e 20001 com duas ocorrências |
| Quais pagamentos não encontram pedido? | Pagamentos `left_anti` pedidos | 25 registros, incluindo as duplicidades observadas |

A semântica do join determina quais correspondências e ausências entram no resultado; cardinalidade determina quantas linhas elas produzem. `BroadcastHashJoin` usa uma relação hash transmitida aos executors, enquanto o `SortMergeJoin` salvo alinha ambos os lados por shuffle e os ordena dentro das partições. A escolha física não substitui validação das chaves e não permite concluir desempenho sem medições.

A chamada isolada a `F.broadcast(df_clientes)` não alterou retroativamente o join já construído. Os planos com `BuildLeft` ou `BuildRight` mostram qual entrada é transmitida em cada operador; a posição pode mudar entre consultas.

Consulte [o capítulo de joins](../docs/07_joins.md) e [o Desafio 07](../desafios/desafio_07_joins.ipynb) para continuar o estudo, incluindo o exercício de chave composta presente no desafio.